# QuantForge -- RL Agent Training (Phase 7 part A)

Trains a `stable-baselines3` **DQN** policy on the `TradingEnv` Gymnasium
environment (`ml/envs/trading_env.py`), which wraps the backend's own
feature-engineering code (`backend/app/strategy_engine/features.py`) --
the same function that will drive live inference once Phase 7 part B
wires the trained checkpoint into an `RLStrategy`. Train-time and
serve-time features can never drift apart, because there is only one
implementation of "the state".

**Design choices already made in code (see CLAUDE.md's "RL Environment"
section for the full rationale) -- this notebook just trains against them:**

- **Action space: `Discrete(3)` (HOLD / BUY / SELL), trained with DQN** --
  not a continuous position-delta + PPO. This mirrors the project's
  existing `SignalAction` vocabulary and long/flat-only position model,
  so the trained policy's output slots directly into the same
  `Signal` / `RiskManager.evaluate` pipeline every other strategy uses.
- **Reward: risk-adjusted (Sharpe-shaped) return** -- realized next-bar
  return minus a transaction-cost penalty on position changes, divided
  by trailing realized volatility.
- **Chronological 70/15/15 train/val/test split, never shuffled** --
  `envs.trading_env.chronological_split`.
- **Getting the code into Colab: git clone + `sys.path`, not
  `pip install -e`.** The repo isn't packaged (no `setup.py`/
  `pyproject.toml` on `backend/` or `ml/`, and this project doesn't
  otherwise need one) -- cloning and adding `backend/` and `ml/` to
  `sys.path` gets `app.strategy_engine.features` and `envs.trading_env`
  importable with zero packaging work, which is the simpler of the two
  options for a project this size.

**What this notebook does:** installs dependencies, mounts Google
Drive, clones this repo, fetches historical OHLCV for one symbol using
the backend's own `fetch_candles`, splits it chronologically, trains
DQN on the train split, evaluates on val *and* held-out test, saves the
checkpoint (+ a metadata sidecar) to Drive, and prints exactly what to
download and where it goes in the repo.

**Stop here and hand back to Claude Code** once you've downloaded the
checkpoint -- Phase 7 part B (wiring it into a live `RLStrategy`) picks
up from `ml/checkpoints/`.


## 1. Install dependencies

In [ ]:
# Deliberately NOT `pip install -r backend/requirements.txt` -- that
# pulls in nsepy and backtrader too, and nsepy's setup.py (unmaintained
# since ~2018) has been observed forcing pip to downgrade numpy below
# 2.0 in a fresh Colab runtime, which then breaks half of Colab's own
# preinstalled ML stack (torch/jax/opencv/etc. all require numpy>=2).
# Neither nsepy's NSE fallback path nor backtrader is touched by
# anything this notebook imports (nsepy is only ever imported lazily,
# inside a function, when the yfinance fetch fails -- see
# app/data_service/sources.py), so this installs only what
# `app.strategy_engine.features` / `app.data_service.sources` actually
# need at import time, pinned to the same versions backend/requirements.txt
# uses. Colab's own numpy/pandas (already numpy>=2, pandas>=2) are left
# alone rather than reinstalled.
!pip install -q \
    "pandas-ta==0.4.71b0" \
    "sqlalchemy==2.0.36" \
    "yfinance==1.7.0" \
    "pydantic==2.10.4" \
    "pydantic-settings==2.7.1"
# Training-only deps (not part of the backend service).
!pip install -q -r {REPO_DIR}/ml/requirements.txt


In [ ]:
REPO_URL = "https://github.com/Haridasan417/quantforge.git"
REPO_DIR = "/content/quantforge"

import os
if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q


In [ ]:
# Deliberately NOT `pip install -r backend/requirements.txt` -- that
# pulls in nsepy and backtrader too, and nsepy's setup.py (unmaintained
# since ~2018) has been observed forcing pip to downgrade numpy below
# 2.0 in a fresh Colab runtime, which then breaks half of Colab's own
# preinstalled ML stack (torch/jax/opencv/etc. all require numpy>=2).
# Neither nsepy's NSE fallback path nor backtrader is touched by
# anything this notebook imports (nsepy is only ever imported lazily,
# inside a function, when the yfinance fetch fails -- see
# app/data_service/sources.py), so this installs only what
# `app.strategy_engine.features` / `app.data_service.sources` actually
# need at import time, pinned to the same versions backend/requirements.txt
# uses. Colab's own numpy/pandas (already numpy>=2, pandas>=2) are left
# alone rather than reinstalled -- reinstalling them is exactly what
# triggers the conflict cascade above.
!pip install -q \
    "pandas-ta==0.4.71b0" \
    "sqlalchemy==2.0.36" \
    "yfinance==1.7.0" \
    "pydantic==2.10.4" \
    "pydantic-settings==2.7.1"
# Training-only deps (not part of the backend service).
!pip install -q -r {REPO_DIR}/ml/requirements.txt


In [ ]:
import sys

for path in (f"{REPO_DIR}/backend", f"{REPO_DIR}/ml"):
    if path not in sys.path:
        sys.path.insert(0, path)

# Sanity check -- both should import cleanly with no errors before we go further.
from app.strategy_engine.features import FEATURE_WINDOW, OBSERVATION_SIZE
from envs.trading_env import TradingEnv, chronological_split

print("FEATURE_WINDOW:", FEATURE_WINDOW, " OBSERVATION_SIZE:", OBSERVATION_SIZE)


## 2. Mount Google Drive

The trained checkpoint (and its metadata sidecar) are saved here so they survive the Colab runtime being recycled.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/quantforge_checkpoints"
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", CHECKPOINT_DIR)


## 3. Fetch historical data

Uses the backend's own `fetch_candles` (yfinance-primary, nsepy-fallback
for `.NS` symbols) -- the exact same fetch code `/api/candles` and the
backtester use, not a separate notebook-only data pull. Edit `SYMBOL`/
`START`/`END` to train on a different symbol or range.

In [ ]:
from datetime import datetime, timezone

from app.data_service.sources import fetch_candles

SYMBOL = "RELIANCE.NS"
INTERVAL = "1d"
START = datetime(2015, 1, 1, tzinfo=timezone.utc)
END = datetime.now(timezone.utc)

raw_df = fetch_candles(SYMBOL, INTERVAL, START, END)
print(f"Fetched {len(raw_df)} bars for {SYMBOL}: {raw_df.index[0].date()} -> {raw_df.index[-1].date()}")
raw_df.tail()


## 4. Chronological train / val / test split (70 / 15 / 15)

Never shuffled -- see `chronological_split`'s docstring.

In [ ]:
train_df, val_df, test_df = chronological_split(raw_df, train_frac=0.7, val_frac=0.15)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s}: {len(split):4d} bars  {split.index[0].date()} -> {split.index[-1].date()}")


## 5. Train DQN on the train split

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor

train_env = Monitor(TradingEnv(train_df))

TOTAL_TIMESTEPS = 100_000  # ~ a few dozen passes over a multi-year daily-bar train split; raise if the reward curve hasn't leveled off yet

model = DQN(
    "MlpPolicy",
    train_env,
    verbose=1,
    learning_rate=1e-4,
    buffer_size=50_000,
    learning_starts=1_000,
    batch_size=64,
    gamma=0.99,
    train_freq=4,
    target_update_interval=1_000,
)

model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)


## 6. Evaluate on validation, then held-out test

`evaluate_policy` reports mean/std episode reward (the env's own
risk-adjusted reward, summed over one full pass through the split --
each split is one episode start to finish). A quick cumulative
raw-return readout is printed alongside it, since "mean reward" alone
doesn't say much about whether the policy actually made money.

In [ ]:
import numpy as np
from stable_baselines3.common.evaluation import evaluate_policy


def evaluate_split(name: str, df):
    env = TradingEnv(df)
    mean_reward, std_reward = evaluate_policy(model, Monitor(env), n_eval_episodes=1, deterministic=True)

    # A second, deterministic pass to report cumulative raw (un-shaped)
    # return -- easier to sanity-check by eye than the Sharpe-shaped
    # training reward.
    obs, _ = env.reset(seed=0)
    terminated = False
    cumulative_raw_return = 0.0
    while not terminated:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(int(action))
        cumulative_raw_return += info["raw_reward"]

    print(f"{name:5s}  mean_reward={mean_reward:.4f}  std_reward={std_reward:.4f}  cumulative_raw_return={cumulative_raw_return:.4%}")
    return {"mean_reward": float(mean_reward), "std_reward": float(std_reward), "cumulative_raw_return": float(cumulative_raw_return)}


val_metrics = evaluate_split("val", val_df)
test_metrics = evaluate_split("test", test_df)


## 7. Save the checkpoint (+ metadata sidecar) to Drive

In [ ]:
import json
from datetime import datetime, timezone

trained_at = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
checkpoint_name = f"dqn_{SYMBOL.replace('.', '_')}_{trained_at}"
checkpoint_path = f"{CHECKPOINT_DIR}/{checkpoint_name}.zip"
metadata_path = f"{CHECKPOINT_DIR}/{checkpoint_name}.json"

model.save(checkpoint_path)

metadata = {
    "checkpoint_name": checkpoint_name,
    "algo": "DQN",
    "policy": "MlpPolicy",
    "symbol": SYMBOL,
    "interval": INTERVAL,
    "data_start": str(raw_df.index[0]),
    "data_end": str(raw_df.index[-1]),
    "train_bars": len(train_df),
    "val_bars": len(val_df),
    "test_bars": len(test_df),
    "total_timesteps": TOTAL_TIMESTEPS,
    "trained_at": trained_at,
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
    "feature_window": FEATURE_WINDOW,
    "observation_size": OBSERVATION_SIZE,
}
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved checkpoint to:", checkpoint_path)
print("Saved metadata to:  ", metadata_path)
print()
print(json.dumps(metadata, indent=2))


## 8. Download the checkpoint

Two ways to get the files onto your machine:

**Option A -- from the Drive web UI (simplest):** open
[drive.google.com](https://drive.google.com/), navigate to
`quantforge_checkpoints/`, and download both the `.zip` and the
matching `.json` printed above.

**Option B -- direct browser download from this notebook:** run the
cell below.

In [ ]:
from google.colab import files

files.download(checkpoint_path)
files.download(metadata_path)


## Next step (outside this notebook)

Move both downloaded files into the repo at **`ml/checkpoints/`**
(create it if it doesn't exist locally -- it's already tracked as an
empty directory placeholder), keeping the matching `.zip`/`.json` pair
together, e.g.:

```
ml/checkpoints/dqn_RELIANCE_NS_20260910T120000Z.zip
ml/checkpoints/dqn_RELIANCE_NS_20260910T120000Z.json
```

Per CLAUDE.md's free-tier note: only this small JSON metadata is meant
to live in the repo/DB long-term -- if the `.zip` itself gets large
enough to strain a normal git push, switch to Git LFS or commit only
the metadata and keep the binary in Drive/a release asset instead.

Then hand back to Claude Code for **Phase 7 part B**: wiring this
checkpoint into a live `RLStrategy` (load the model, call
`build_observation` per bar, map the discrete action back to a
`Signal`) for backtesting and paper-trading execution.
